# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 12 — Deep Learning: Exercise Solutions</div>
<div align="center"> Jonathan Holmes (he/him)</div>

These exercises accompany the *Class 12 — Deep Learning* lecture notebook. They cover the architecture of a small feedforward network, parameter counting, the role of nonlinear activations, model choice between OLS / Lasso / neural nets, and how bias and variance respond to architectural changes.

# In-Class Exercise #1 — Reading the schematic

**Q1. In the schematic network above, how many input features $(X_1,\ldots,X_4)$ are there? How many hidden units $(A_1,\ldots,A_5)$?**

---

**Answer.**

The naming is the giveaway: $(X_1,\ldots,X_4)$ means there are **4 input features**, and $(A_1,\ldots,A_5)$ means there are **5 hidden units** in the (single) hidden layer. This is a tiny *feedforward* network with one hidden layer of width 5 and a one-dimensional output.

**Q2. Count trainable parameters (weights + biases):**
* **For each hidden neuron $h_k$: how many weights and how many biases?**
* **If the output is a linear function of $(A_1,\ldots,A_5)$, e.g. $f(X) = \beta_0 + \beta_1 A_1 + \cdots + \beta_5 A_5$, how many parameters in the output layer?**
* **Total parameters in this toy network?**

---

**Answer.**

Each hidden neuron $h_k$ takes the four inputs and computes
$$A_k = g\!\left(w_{k1}X_1 + w_{k2}X_2 + w_{k3}X_3 + w_{k4}X_4 + b_k\right),$$
so it has **4 weights and 1 bias = 5 parameters**.

Adding up the layers:

| Layer | Parameters per unit | Number of units | Subtotal |
|-------|--------------------|-----------------|---------|
| Hidden layer | $4 + 1 = 5$ | $5$ | $25$ |
| Output layer | $5 + 1 = 6$ | $1$ | $6$ |
| **Total** | | | **$31$** |

So the toy network has **31 trainable parameters**. Two things are worth noticing:

1. Even this small network already has more free parameters than a multiple linear regression of $Y$ on $(X_1,\ldots,X_4)$ (which would have $4 + 1 = 5$).
2. Parameter count grows quickly. With $p$ inputs, $H$ hidden units, and 1 output unit you have $H(p + 1) + (H + 1)$ parameters — that scales as $H \cdot p$. Modern deep networks easily reach millions or billions of parameters, which is why they need lots of data and regularization.

**Q3. Why stack nonlinear activations $g$ instead of leaving every unit purely linear?**

---

**Answer.**

Because the **composition of linear functions is itself linear**. If every hidden unit just computed $A_k = w_k^\top X + b_k$ (no nonlinearity), then the output is a linear combination of linear combinations of $X$ — which is again a single linear function of $X$. The network of any depth would collapse to a plain linear regression, with no more representational power than what we did in Class 6.

Concretely, suppose two hidden layers are stacked with no activation: $A^{(1)} = W_1 X + b_1$, $A^{(2)} = W_2 A^{(1)} + b_2$. Then
$$A^{(2)} = W_2 W_1 X + (W_2 b_1 + b_2) = \tilde W X + \tilde b,$$
which is a single affine map.

Inserting a *nonlinear* $g$ (sigmoid, ReLU, tanh, …) breaks that collapse and lets each layer represent something the layer below cannot. With nonlinear activations, a sufficiently wide one-hidden-layer network is a *universal function approximator* — it can approximate any continuous function arbitrarily well. Without them, the network is just OLS in disguise.

# In-Class Exercise #2 — Choosing a model

**Q4. You want good out-of-sample predictions and are choosing among OLS, Lasso, and a neural net. How do you decide?**

---

**Answer.**

Frame the choice in terms of the bias–variance tradeoff and what each method buys you:

* **OLS** — low variance, but high bias if the true relationship is nonlinear or includes interactions you have not specified. Best when $n \gg p$, the relationship really is roughly linear, and interpretability matters. Cheap to fit and easy to explain.
* **Lasso** — handles many predictors gracefully ($p$ comparable to or larger than $n$), automatically selects a sparse subset, and stabilizes predictions when predictors are correlated. Still linear, so it cannot capture nonlinearities or interactions you have not engineered into the design matrix.
* **Neural network** — very flexible (low bias) but high variance: needs lots of data to train without overfitting, careful regularization (dropout, early stopping, weight decay), and computational budget for hyperparameter tuning. Best when the true relationship is complex, $n$ is large, and you do not need to explain individual coefficients.

**The decision in practice:**

1. Start with the simplest model that could plausibly work — usually OLS or Lasso.
2. Use **cross-validation** to compare candidates on the *same* test-error metric. The model with the lowest CV error wins.
3. Break ties using non-statistical criteria: interpretability, deployment cost, regulatory requirements.

If $n$ is small, a neural net will almost always lose to a regularized linear model on tabular data; the flexibility you are paying for is variance you cannot afford. If $n$ is large and the relationship is genuinely nonlinear, a tuned neural net (or a boosted tree ensemble) often wins.

**Q5. Same task, but 100× more training data. What changes?**

---

**Answer.**

The bias–variance balance shifts toward methods with **lower bias** (and higher variance), because variance is now cheap to control with data:

* **OLS**'s coefficient estimates become very precise, but its bias does not move — if the true relationship is nonlinear, OLS still systematically misses, no matter how much data you throw at it.
* **Lasso** also gains precision; the selected subset becomes more stable and the optimal $\lambda$ tends to drop (less shrinkage needed).
* **Neural networks** benefit the most. With 100× more data, an over-parameterized network has far less room to overfit — the noise effectively averages out — so the network can use its full flexibility to chase the (possibly nonlinear) true relationship. The variance penalty that hurt it on the small dataset largely disappears.

Intuitively, more data shrinks the variance term in $\text{Test MSE} = \text{Bias}^2 + \text{Variance} + \text{Irreducible noise}$, so the *bias* term increasingly dominates. Methods that have low bias by construction (flexible models) close in on the irreducible-noise floor, while inflexible methods bottom out at their bias.

Empirically: the rank of methods on tabular data often *changes* as $n$ grows — a small-data winner (Lasso, OLS) may be overtaken by trees, boosting, or neural nets once the sample is large enough to feed them.

# In-Class Exercise #3 — Architecture and the bias–variance tradeoff

**Q6. Holding data and training procedure fixed, if you add layers (more flexibility), you usually expect bias to ___ and variance to ___ ?**

---

**Answer.**

Bias **decreases**; variance **increases**.

Adding layers makes the network capable of representing richer, more deeply nested functions (the *function class* gets bigger). That richer class can come closer to the true $f(X)$, so systematic error — bias — falls. But a richer class also has more free parameters, more optima, and more ways to fit idiosyncratic noise in the training set, so variance rises.

The total test error $\text{Bias}^2 + \text{Variance} + \text{Irreducible noise}$ traces out a U: too few layers underfits (bias dominates), too many layers overfits (variance dominates), and the right depth depends on $n$ and the regularization in use. This is exactly the same logic we saw with $K$ in KNN or polynomial degree in regression — just with depth as the complexity dial.

**Q7. Same question if you add neurons per layer instead.**

---

**Answer.**

Same direction: bias **decreases**, variance **increases**. *Width* (units per layer) is another way to add flexibility, with the same bias–variance consequences as *depth* (number of layers).

The folk wisdom — and a fair empirical generalization for tabular problems — is that **depth and width are partially substitutable**, but not perfectly:

* Adding **width** with fixed depth grows representational capacity roughly linearly. It is often the safer first move because wider but shallow networks are easier to train.
* Adding **depth** can capture compositional / hierarchical structure that wide-but-shallow networks struggle with — but deep networks are harder to train (vanishing gradients, slower convergence) and tend to need more careful regularization.

Either way, both are knobs on the same dial: more parameters → more flexibility → lower bias, higher variance, more risk of overfitting on a fixed training set. The right architecture is whatever cross-validation says generalizes best.